In [ ]:
import sys
sys.path.append("..")   # add main_folder to path


#Import libaries
from pathlib import Path
import geopandas
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Point
import geodatasets
import seaborn as sns
import numpy as np
import tomllib
import re
import pyarrow.dataset as pds
import pyarrow.compute as pc
import pyarrow as pa

from src.genesis.genesis_utils import read_ibtracs, preprocess_ibtracs
#from src.data.load_ibtracks import read_ibtracs,full_processing

#Helper 
def skip_row_func(row_number):
    if (
        row_number == 1
    ):  # skip second row since it contain units, to read the types properly
        return True
    return False

catherina_fit_path = Path(
    "../../data/input/fit/Catherina_fit.db"
)

method = "RMSF"
config_path =  "../config.toml"

with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

In [ ]:
SAFFIR_SIM_CAT_MS = [32.92, 42.7, 49.3, 58.13, 70.47, 999]
CATEGORY_LABELS = ["Category 1", "Category 2", "Category 3", "Category 4", "Category 5"]

def get_category(wind_speed):
    if wind_speed >= 70:
        return '5'
    elif wind_speed >= 58:
        return '4'
    elif wind_speed >= 50:
        return '3'
    elif wind_speed >= 43:
        return '2'
    elif wind_speed >= 33:
        return '1'
    elif wind_speed >= 0:
        return 'TS'  # Tropical Storm
    
#### Catherina proc
def compute_freq_cath(tracks):
    """
    Compute the frequency of storms in different basins based on wind speed.

    Parameters:
    - tracks: DataFrame containing storm data.
              Expected columns include 'ISO_TIME_POSIXct', 'BASIN', 'wind', 'SID', and 'seed'.

    Returns:
    - df_mean: DataFrame with the mean frequency for each basin.
    - df_std: DataFrame with the standard deviation of frequency for each basin.
    """

    # Extract year from 'ISO_TIME_POSIXct' and filter based on wind speed
    tracks["year"] = pd.to_datetime(tracks["datetime"]).dt.year
    tracks["basin"] = tracks["basin"].fillna("NA")
    strong_wind_tracks = tracks[tracks["final_wind_speed"] > 32.92]

    # Group by SID, seed, and BASIN to get max wind for each group
    max_wind_df = (
        strong_wind_tracks.groupby(["SID", "seed", "basin"])["final_wind_speed"].max().reset_index()
    )

    # Initialize stats list to store frequency data
    stats_list = []
    basins = max_wind_df["basin"].unique()
    seeds = max_wind_df["seed"].unique()

    # Compute the normalized frequency for each basin and seed combination
    for basin in basins:
        for seed in seeds:
            subset = max_wind_df[
                (max_wind_df["basin"] == basin) & (max_wind_df["seed"] == seed)
            ]
            freq, _ = np.histogram(subset["final_wind_speed"], bins=SAFFIR_SIM_CAT_MS, density=True)
            freq_normed = freq / freq.sum()
            stats_list.append({"basin": basin, "seed": seed, "freq": freq_normed})

    # Convert stats list to DataFrame
    df_stats = pd.DataFrame(stats_list)

    # Compute mean and standard deviation for each basin
    df_mean = (
        df_stats.groupby("basin")
        .apply(lambda group: group["freq"].apply(pd.Series).mean())
        .reset_index()
    )
    df_std = (
        df_stats.groupby("basin")
        .apply(lambda group: group["freq"].apply(pd.Series).std())
        .reset_index()
    )

    return df_mean, df_std

def compute_storm_frequency(final_storms_df):
    """
    Computes the storm frequency for each basin and category.
    """

    max_w = final_storms_df.groupby('SID')["wind"].transform("max")
    final_storms_df = final_storms_df[final_storms_df["wind"].eq(max_w)].sort_values('SID')  # keeps every max-tie row

    # Categorize storms based on wind speed
    final_storms_df["category"] = pd.cut(
        final_storms_df["wind"],
        bins=SAFFIR_SIM_CAT_MS,
        labels=CATEGORY_LABELS,
        right=False,
        include_lowest=True,
    )

    # Calculate frequency for each basin and category
    all_basins, all_categories, all_frequencies = [], [], []
    for basin in final_storms_df["BASIN"].unique():
        basin_subset = final_storms_df[final_storms_df["BASIN"] == basin]
        freq, _ = np.histogram(
            basin_subset["wind"], bins=SAFFIR_SIM_CAT_MS, density=True
        )
        normed_freq = freq / freq.sum()
        all_basins.extend([basin] * len(CATEGORY_LABELS))
        all_categories.extend(CATEGORY_LABELS)
        all_frequencies.extend(normed_freq)

    # Create the output dataframe
    result_df = pd.DataFrame(
        {"BASIN": all_basins, "category": all_categories, "freq": all_frequencies}
    )

    return result_df

In [ ]:
#Read and preprocess ibtracks
main_config = config_files["main_params"]
gen_config = config_files["generation"]
data_dir = ".." / Path(main_config["input_data_dir"])
ibtracksf_folder = data_dir / gen_config["ibtracs_path"]

#Theo version
#ibtracs_theo = process_ibtracs_data(ibtracksf_folder,2014, 32.92)

#New version
ibtracs = read_ibtracs(fpath=ibtracksf_folder, signed_coords=True)
ibtracs = preprocess_ibtracs(ibtracs, 32.92)
#ibtracs['year'] = ibtracs['ISO_TIME_POSIXct'].dt.year
#ibtracs = ibtracs.loc[lambda row:(row['year']>=1990)&(row['year']<2015),:]
# ibtracs = process_ibtracs_data(ibtracksf_folder)




In [ ]:
# Load tracks data

tracks_historical_dir = Path("../../data/input/catherina_historical/intensified_tracks_test_base_param/ACCESS-CM2/historical/")
dataset = pds.dataset(tracks_historical_dir, 
                      format="parquet",
                        partitioning="hive")
part_fields = {f.name: f.type for f in dataset.partitioning.schema}

#year_f = pc.cast(pc.field("year"), pa.int32()) if part_fields.get("year") == pa.string() else pc.field("year")
#month_f = pc.cast(pc.field("month"), pa.int32()) if part_fields.get("month") == pa.string() else pc.field("month")
seed_f = pc.cast(pc.field("seed"), pa.int64())  if part_fields.get("seed") == pa.string() else pc.field("seed")

filt = seed_f.isin(list(range(50)))

#Avoid validity checks in the Scanner if columns may be absent; filter later
scanner = pds.Scanner.from_dataset(dataset, filter=filt)

tracks_historical = scanner.to_table().to_pandas().reset_index()


#Filter out some tracks:
tracks_historical = tracks_historical.sort_values(["SID", "datetime"], kind="mergesort").copy()
#Condition on max wind
max_wind_df = tracks_historical.groupby("SID")["final_wind_speed"].max()
min_sst_df = tracks_historical.groupby("SID")["SST"].min()
# filtering out storms with wind speed inferior to threshold
tracks_to_keep = max_wind_df[(max_wind_df >= 32.92)&(max_wind_df <= 100)&(min_sst_df>10)].index
filtered_tracks_historical = tracks_historical.loc[tracks_historical["SID"].isin(tracks_to_keep)]


In [ ]:
import seaborn as sns
sns.lineplot(filtered_tracks_historical.loc[lambda row:(row['SID']==2614861455)&(row['seed']==1),'final_wind_speed'])

In [ ]:
ibtracs_freq_all = compute_storm_frequency(ibtracs)
stats, std_dev = compute_freq_cath(filtered_tracks_historical)

std_dev.iloc[:, -1] = 0


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))
axes = axes.ravel()

for i, basin in enumerate(stats["basin"]):
    freq_p1 = stats.iloc[i, 1:].values
    error_p1 = std_dev.iloc[i, 1:].values
    ibtracs_freq = ibtracs_freq_all[ibtracs_freq_all["BASIN"] == basin]["freq"].values

    # X coordinates for categories
    x_coords = np.arange(len(CATEGORY_LABELS))

    # Plot bars for all categories except Category 5 on the primary Y axis
    axes[i].bar(
        x_coords[:-1] - 0.15,
        ibtracs_freq[:-1],
        alpha=0.7,
        width=0.30,
        color="green",
        label=f"{basin} Ibtracs",
    )
    axes[i].bar(
        x_coords[:-1] + 0.15,
        freq_p1[:-1],
        yerr=error_p1[:-1],
        alpha=0.7,
        capsize=5,
        width=0.30,
        color='blue',
        label=f"{basin} ACCESS-CM2",
    )

    axes[i].set_ylim(bottom=0)

    # Create a secondary Y axis
    ax2 = axes[i].twinx()

    # Plot bars for Category 5 on the secondary Y axis without labels
    ax2.bar(
        x_coords[-1] - 0.15, ibtracs_freq[-1], alpha=0.7, width=0.30, color="darkgreen"
    )
    ax2.bar(
        x_coords[-1] + 0.15,
        freq_p1[-1],
        yerr=error_p1[-1],
        alpha=0.7,
        capsize=5,
        width=0.30,
        color="darkblue",
    )

    # Set title and X-axis label for the primary Y axis
    axes[i].set_title(f"Cyclone Frequencies in {basin}")
    axes[i].set_xlabel("Category")
    axes[i].set_ylabel("Frequency")

    # Set Y-axis label for the secondary Y axis
    ax2.set_ylabel("Frequency (Cat 5)", color="darkblue")

    # Changing the color of the Y-axis labels to match the bars' color
    ax2.tick_params(axis="y", colors="darkblue")
    ax2.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    # Add a vertical line to separate Category 5 from the rest
    axes[i].axvline(x=x_coords[-1] - 0.5, color="gray", linestyle="--", lw=1)

    # Setting the legend (only considering the primary axis)
    lines, labels = axes[i].get_legend_handles_labels()
    axes[i].legend(lines, labels, loc="upper left")

    axes[i].set_xticks(x_coords)
    axes[i].set_xticklabels(CATEGORY_LABELS)
    axes[i].tick_params(axis="x", rotation=45)

    #axes[-1].axis("off")  # remove the last plot

plt.tight_layout()
plt.show()

In [ ]:
# Build dataset
dataset = pds.dataset(Path("../../data/input/catherina_ssp585/intensified_tracks_test/ACCESS-CM2/ssp585/"), format="parquet", partitioning="hive")

# Partition field types may be strings; cast to numeric for comparisons
part_fields = {f.name: f.type for f in dataset.partitioning.schema}

def _cast_part(name, want_type):
    f = pc.field(name)
    return pc.cast(f, want_type) if part_fields.get(name) == pa.string() else f

seed_f = _cast_part("seed", pa.int64())
year_f = _cast_part("year", pa.int32())

# Base filter: seed < 100
base = seed_f < pa.scalar(20, pa.int64())

# Year slices
filt_2025_2050 = base & (year_f < pa.scalar(2050, pa.int32()))
filt_2050_2075 = base & (year_f >= pa.scalar(2050, pa.int32())) & (year_f < pa.scalar(2075, pa.int32()))
filt_2075_2100 = base & (year_f >= pa.scalar(2075, pa.int32()))

# Only pull the columns you need (plus partition cols)
cols = [
    "SID", "step", "SST","lat_left", "lon_left", "datetime", "final_wind_speed", "basin",
    "seed", "year"
]

def _scan_to_df(filt):
    scanner = pds.Scanner.from_dataset(
        dataset, filter=filt, columns=cols, use_threads=True, batch_size=1<<18
    )
    tbl = scanner.to_table()
    df = tbl.to_pandas(types_mapper=pd.ArrowDtype)  # fast, preserves dtypes
    df = df.reset_index()
    #Filter out some tracks:
    df = df.sort_values(["SID", "datetime"], kind="mergesort").copy()

    #Condition on max wind
    max_wind_df = df.groupby("SID")["final_wind_speed"].max()
    # filtering out storms with wind speed inferior to threshold
    tracks_to_keep = max_wind_df[(max_wind_df >= 32.92)&(max_wind_df <= 100)].index
    df_filtered = df.loc[df["SID"].isin(tracks_to_keep)]


    return df_filtered

# # Read the three slices
tracks_ssp585_2025_2050 = _scan_to_df(filt_2025_2050)
tracks_ssp585_2050_2075 = _scan_to_df(filt_2050_2075)
tracks_ssp585_2075_2100 = _scan_to_df(filt_2075_2100)

In [ ]:
tracks_ssp585_2025_2050 = tracks_ssp585_2025_2050.loc[lambda row:(~row['final_wind_speed'].isna())&(row['basin']!="SA")]
tracks_ssp585_2050_2075 = tracks_ssp585_2050_2075.loc[lambda row:~row['final_wind_speed'].isna()&(row['basin']!="SA")]
tracks_ssp585_2075_2100 = tracks_ssp585_2075_2100.loc[lambda row:~row['final_wind_speed'].isna()&(row['basin']!="SA")]

In [ ]:
ibtracs_freq_all = compute_storm_frequency(ibtracs)
stats_p1, std_dev_p1 = compute_freq_cath(tracks_ssp585_2025_2050)
stats_p2, std_dev_p2 = compute_freq_cath(tracks_ssp585_2050_2075)
stats_p3, std_dev_p3 = compute_freq_cath(tracks_ssp585_2075_2100)


std_dev_p1.iloc[:, -1] = 0
std_dev_p2.iloc[:, -1] = 0
std_dev_p3.iloc[:, -1] = 0

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))
axes = axes.ravel()

for i, basin in enumerate(stats_p1["basin"]):
    freq_p1 = stats_p1.iloc[i, 1:].values
    error_p1 = std_dev_p1.iloc[i, 1:].values
    freq_p2 = stats_p2.iloc[i, 1:].values
    error_p2 = std_dev_p2.iloc[i, 1:].values
    freq_p3 = stats_p3.iloc[i, 1:].values
    error_p3 = std_dev_p3.iloc[i, 1:].values
    ibtracs_freq = ibtracs_freq_all[ibtracs_freq_all["BASIN"] == basin]["freq"].values

    # X coordinates for categories
    x_coords = np.arange(len(CATEGORY_LABELS))

    # Plot bars for all categories except Category 5 on the primary Y axis
    axes[i].bar(
        x_coords[:-1] - 0.30,
        ibtracs_freq[:-1],
        alpha=0.7,
        width=0.15,
        color="green",
        label=f"{basin} Ibtracs",
    )
    axes[i].bar(
        x_coords[:-1] - 0.15,
        freq_p1[:-1],
        yerr=error_p1[:-1],
        alpha=0.7,
        capsize=5,
        width=0.15,
        color='blue',
        label=f"{basin} ACCESS-CM2 (st)",
    )
    axes[i].bar(
        x_coords[:-1] + 0.15,
        freq_p2[:-1],
        yerr=error_p2[:-1],
        alpha=0.7,
        capsize=5,
        width=0.15,
        color='red',
        label=f"{basin} ACCESS-CM2 (mt)",
    )
    axes[i].bar(
        x_coords[:-1] + 0.30,
        freq_p3[:-1],
        yerr=error_p3[:-1],
        alpha=0.7,
        capsize=5,
        width=0.15,
        color='yellow',
        label=f"{basin} ACCESS-CM2 (lt)",
    )


    axes[i].set_ylim(bottom=0)

    # Create a secondary Y axis
    ax2 = axes[i].twinx()

    # Plot bars for Category 5 on the secondary Y axis without labels
    ax2.bar(
        x_coords[-1] - 0.3, ibtracs_freq[-1], alpha=0.7, width=0.15, color="darkgreen"
    )
    ax2.bar(
        x_coords[-1] - 0.15,
        freq_p1[-1],
        yerr=error_p1[-1],
        alpha=0.7,
        capsize=5,
        width=0.15,
        color="darkblue",
    )
    ax2.bar(
        x_coords[-1] + 0.15,
        freq_p2[-1],
        yerr=error_p2[-1],
        alpha=0.7,
        capsize=5,
        width=0.15,
        color="darkred",
    )
    ax2.bar(
        x_coords[-1] + 0.30,
        freq_p3[-1],
        yerr=error_p3[-1],
        alpha=0.7,
        capsize=5,
        width=0.15,
        color="yellow",
    )

    # Set title and X-axis label for the primary Y axis
    axes[i].set_title(f"Cyclone Frequencies in {basin}")
    axes[i].set_xlabel("Category")
    axes[i].set_ylabel("Frequency")

    # Set Y-axis label for the secondary Y axis
    ax2.set_ylabel("Frequency (Cat 5)", color="darkblue")

    # Changing the color of the Y-axis labels to match the bars' color
    ax2.tick_params(axis="y", colors="darkblue")
    ax2.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    # Add a vertical line to separate Category 5 from the rest
    axes[i].axvline(x=x_coords[-1] - 0.5, color="gray", linestyle="--", lw=1)

    # Setting the legend (only considering the primary axis)
    lines, labels = axes[i].get_legend_handles_labels()
    axes[i].legend(lines, labels, loc="upper left")

    axes[i].set_xticks(x_coords)
    axes[i].set_xticklabels(CATEGORY_LABELS)
    axes[i].tick_params(axis="x", rotation=45)

    #axes[-1].axis("off")  # remove the last plot

plt.tight_layout()
plt.show()